# 08 — Insulation Dataset Expansion and Split Design

**Purpose:** prepare an evidence-backed collection checklist and investigate group-aware split options for the insulation dataset.

This notebook is planning and simulation only:
- It does not edit the master dataset.
- It does not create synthetic records.
- It does not train or tune ML models.
- It does not automatically approve a split for final evaluation.
- It keeps `split_group_id` as the indivisible grouping unit.

Run with the `arch_ai` kernel, from top to bottom.

## 1. Configuration and robust dataset discovery

The notebook first checks the expected location, then searches within the SpecSynth project folder for the exact dataset filename. If multiple matches are found, it stops rather than selecting an ambiguous version.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(
    r"D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI"
)
DATA_FILENAME = "SpecSynth_ML_variants_v1.5.0.csv"
EXPECTED_DATA_PATH = (
    PROJECT_ROOT / "04_Product_Dataset" / "10_Master_Database" /
    "SpecSynth_Master_Database_v1.5.0" / "source_data" / "v1.5.0" /
    "ml" / DATA_FILENAME
)

ML_ROOT = PROJECT_ROOT / "06_AI_Performance_Engine" / "02_ML_Engine"
OUTPUT_DIR = ML_ROOT / "04_Evaluation" / "reports" / "insulation_expansion_split_design_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if EXPECTED_DATA_PATH.exists():
    DATA_PATH = EXPECTED_DATA_PATH
else:
    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(
            f"Project root not found: {PROJECT_ROOT}\n"
            "Update PROJECT_ROOT to your actual SpecSynth_AI folder."
        )
    matches = list(PROJECT_ROOT.rglob(DATA_FILENAME))
    if len(matches) == 1:
        DATA_PATH = matches[0]
    elif len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {DATA_FILENAME} inside {PROJECT_ROOT}."
        )
    else:
        raise RuntimeError(
            "Multiple dataset versions/locations found. Select the intended file manually:\n"
            + "\n".join(str(p) for p in matches)
        )

print("Dataset:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)


Dataset: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI\04_Product_Dataset\10_Master_Database\SpecSynth_Master_Database_v1.0\source_data\v1.5.0\ml\SpecSynth_ML_variants_v1.5.0.csv
Output directory: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI\06_AI_Performance_Engine\02_ML_Engine\04_Evaluation\reports\insulation_expansion_split_design_v1


**Conclusion:** verify that the printed path points to the intended v1.5.0 source file. The notebook deliberately refuses to guess if more than one exact match exists.

## 2. Load and validate the insulation records

In [2]:
df_all = pd.read_csv(DATA_PATH)
required = {
    "material_category_group", "product_id", "variant_id",
    "split_group_id", "material_family", "thermal_conductivity_W_mK"
}
missing = required - set(df_all.columns)
if missing:
    raise ValueError(f"Required columns missing: {sorted(missing)}")

df = df_all.loc[
    df_all["material_category_group"].eq("A_thermal_insulation")
].copy()

print("All-material records:", len(df_all))
print("Insulation records:", len(df))
print("Products:", df["product_id"].nunique())
print("Variants:", df["variant_id"].nunique())
print("Independent groups:", df["split_group_id"].nunique())
print("Families:", df["material_family"].nunique())

if df["split_group_id"].isna().any():
    raise ValueError("Missing split_group_id values detected. Resolve before split simulation.")


All-material records: 1941
Insulation records: 1674
Products: 70
Variants: 1674
Independent groups: 69
Families: 16


**Conclusion:** missing group IDs must be resolved before any group-aware split can be trusted. Do not replace missing IDs with row numbers; that would falsely treat related records as independent.

## 3. Establish family-level collection gaps

The initial planning target below is **10 independent product groups per included family** and **3 manufacturers where feasible**. These are project planning thresholds, not statistical guarantees. The table shows the additional groups needed to reach the threshold.

In [3]:
TARGET_GROUPS_PER_FAMILY = 10
TARGET_MANUFACTURERS_PER_FAMILY = 3

manufacturer_col = "manufacturer" if "manufacturer" in df.columns else None

agg_spec = {
    "records": ("variant_id", "count"),
    "products": ("product_id", "nunique"),
    "groups": ("split_group_id", "nunique"),
}
if manufacturer_col:
    agg_spec["manufacturers"] = (manufacturer_col, "nunique")

family_plan = (
    df.groupby("material_family", dropna=False)
      .agg(**agg_spec)
      .sort_values(["groups", "products"], ascending=False)
)
family_plan["additional_groups_to_target"] = (
    TARGET_GROUPS_PER_FAMILY - family_plan["groups"]
).clip(lower=0)

if manufacturer_col:
    family_plan["additional_manufacturers_to_target"] = (
        TARGET_MANUFACTURERS_PER_FAMILY - family_plan["manufacturers"]
    ).clip(lower=0)

family_plan["collection_priority"] = np.select(
    [
        family_plan["groups"] <= 2,
        family_plan["groups"] <= 4,
        family_plan["groups"] < TARGET_GROUPS_PER_FAMILY
    ],
    ["P0 — critical support gap", "P1 — low support", "P2 — expand coverage"],
    default="P3 — maintain and diversify"
)
display(family_plan)
family_plan.to_csv(OUTPUT_DIR / "insulation_family_collection_targets_v1.csv")


,records,products,groups,manufacturers,additional_groups_to_target,additional_manufacturers_to_target,collection_priority
material_family,,,,,,,
glass_wool,416,24,24,6,0,0,P3 — maintain and diversify
XPS,179,8,8,4,2,0,P2 — expand coverage
PIR,187,7,7,4,3,0,P2 — expand coverage
stone_wool,151,7,7,4,3,0,P2 — expand coverage
EPS,366,6,6,3,4,0,P2 — expand coverage
wood_wool,28,4,3,4,7,0,P1 — low support
wood_fibre,34,3,3,2,7,1,P1 — low support
cellulose,121,2,2,2,8,1,P0 — critical support gap
polyester_wool,28,2,2,1,8,2,P0 — critical support gap


**Conclusion:** use this table to prioritise collection by independent product group, not by number of catalogue variants. A family with many variants but few products remains a high-priority gap.

## 4. Create a product-level collection checklist

This produces one row per currently represented product. It is a checklist scaffold for researching additional real products; it does not claim that any listed product is missing from the market.

In [4]:
product_cols = ["product_id", "material_family"]
if manufacturer_col:
    product_cols.append(manufacturer_col)

product_checklist = (
    df.groupby(product_cols, dropna=False)
      .agg(
          variant_records=("variant_id", "count"),
          split_groups=("split_group_id", "nunique"),
          conductivity_values=("thermal_conductivity_W_mK", "nunique"),
          conductivity_available=("thermal_conductivity_W_mK", "count")
      )
      .reset_index()
)

for prop in [
    "compressive_strength_kPa", "vapour_resistance_factor_mu",
    "tensile_strength_perpendicular_kPa", "mass_per_area_kg_m2",
    "density_kg_m3", "specific_heat_J_kgK", "gwp_a1_a3_kgco2e_m2"
]:
    if prop in df.columns:
        coverage = df.groupby("product_id")[prop].apply(lambda s: s.notna().any())
        product_checklist[f"has_{prop}"] = product_checklist["product_id"].map(coverage).fillna(False)

product_checklist["research_status"] = "Not yet reviewed"
product_checklist["new_source_document"] = ""
product_checklist["source_url_or_reference"] = ""
product_checklist["evidence_status"] = ""
product_checklist["review_notes"] = ""

display(product_checklist.head(20))
product_checklist.to_csv(
    OUTPUT_DIR / "existing_insulation_product_research_checklist_v1.csv",
    index=False
)


,product_id,material_family,manufacturer,variant_records,split_groups,conductivity_values,conductivity_available,has_compressive_strength_kPa,has_vapour_resistance_factor_mu,has_tensile_strength_perpendicular_kPa,has_mass_per_area_kg_m2,has_density_kg_m3,has_specific_heat_J_kgK,has_gwp_a1_a3_kgco2e_m2,research_status,new_source_document,source_url_or_reference,evidence_status,review_notes
0,AMORIM_ICB_EXPANDED_CORKBOARD,cork,Amorim Cork Solutions,36,1,1,36,True,True,True,False,True,True,False,Not yet reviewed,,,,
1,BAUMIT_PROTHERM_EPS,EPS,Baumit,10,1,1,10,True,False,False,False,True,False,False,Not yet reviewed,,,,
2,CAVAC_JETFIB_OUATE_ACERMI_14_D_130_929,cellulose,CAVAC Biomatériaux,62,1,1,62,False,False,False,True,False,False,False,Not yet reviewed,,,,
3,CELENIT_ACERMI_15_176_1071,wood_wool,CELENIT SPA,7,1,1,7,False,False,False,False,False,False,False,Not yet reviewed,,,,
4,ETEX_MULTITHERM31_ACERMI_26_009_1867,EPS,Etex France Building Performance,57,1,1,57,False,False,True,False,False,False,False,Not yet reviewed,,,,
5,HIRSCH_CELLOCEM_ULTRA_PRIMAIRE_ECA_ACERMI_24_0...,EPS,Hirsch France,77,1,1,77,False,False,True,False,False,False,False,Not yet reviewed,,,,
6,HIRSCH_CELLOMUR_ULTRA_ACERMI_12_081_795,EPS,Hirsch France,78,1,1,78,False,False,True,False,False,False,False,Not yet reviewed,,,,
7,HIRSCH_MAXISOL_ECA_ACERMI_24_081_1679,EPS,Hirsch France,87,1,1,87,True,False,False,False,False,False,False,Not yet reviewed,,,,
8,HIRSCH_STISOL_3000_ECA_ACERMI_24_081_1661,EPS,Hirsch France,57,1,1,57,False,False,False,False,False,False,False,Not yet reviewed,,,,
9,IGLOO_FRANCE_CELLULOSE_ACERMI_12_D_157_784,cellulose,IGLOO France Cellulose SAS,59,1,2,59,False,True,False,True,False,False,False,Not yet reviewed,,,,


**Conclusion:** fill the research columns only after checking actual manufacturer declarations, technical datasheets or other traceable evidence. Do not create placeholder values that could be mistaken for verified data.

## 5. Property collection priorities

This section ranks technical properties by current missingness. Collection priorities should ultimately follow the capabilities SpecSynth intends to support.

In [5]:
property_candidates = [
    "thermal_conductivity_W_mK", "thickness_mm", "thermal_resistance_m2K_W",
    "compressive_strength_kPa", "vapour_resistance_factor_mu",
    "tensile_strength_perpendicular_kPa", "mass_per_area_kg_m2",
    "density_kg_m3", "specific_heat_J_kgK", "gwp_a1_a3_kgco2e_m2"
]
properties = [c for c in property_candidates if c in df.columns]

property_gaps = pd.DataFrame({
    "available_records": df[properties].notna().sum(),
    "missing_records": df[properties].isna().sum(),
    "coverage_pct": (100 * df[properties].notna().mean()).round(2)
})
property_gaps["collection_priority"] = pd.cut(
    property_gaps["coverage_pct"],
    bins=[-0.01, 25, 50, 80, 100.01],
    labels=["P0 — very low coverage", "P1 — low coverage",
            "P2 — moderate coverage", "P3 — high coverage"]
)
display(property_gaps.sort_values("coverage_pct"))
property_gaps.to_csv(OUTPUT_DIR / "insulation_property_collection_priorities_v1.csv")


,available_records,missing_records,coverage_pct,collection_priority
gwp_a1_a3_kgco2e_m2,67,1607,4.00,P0 — very low coverage
specific_heat_J_kgK,115,1559,6.87,P0 — very low coverage
density_kg_m3,163,1511,9.74,P0 — very low coverage
mass_per_area_kg_m2,223,1451,13.32,P0 — very low coverage
tensile_strength_perpendicular_kPa,589,1085,35.19,P1 — low coverage
vapour_resistance_factor_mu,603,1071,36.02,P1 — low coverage
compressive_strength_kPa,607,1067,36.26,P1 — low coverage
thermal_conductivity_W_mK,1614,60,96.42,P3 — high coverage
thermal_resistance_m2K_W,1666,8,99.52,P3 — high coverage
thickness_mm,1674,0,100.00,P3 — high coverage


**Conclusion:** low coverage identifies a data gap, not automatically a useful feature. Collect properties that have a defined role in the intended model and preserve the units, test method, source and evidence status.

## 6. Audit independent group composition

A group should be kept intact across all splits. Inspect whether each group maps cleanly to a single material family and manufacturer before simulating splits.

In [6]:
group_keys = ["split_group_id"]
group_agg = {
    "records": ("variant_id", "count"),
    "products": ("product_id", "nunique"),
    "families": ("material_family", "nunique"),
}
if manufacturer_col:
    group_agg["manufacturers"] = (manufacturer_col, "nunique")

group_table = df.groupby(group_keys).agg(**group_agg)
group_table["family_label"] = (
    df.groupby("split_group_id")["material_family"]
      .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN")
)
group_table["family_is_mixed"] = group_table["families"] > 1
display(group_table.head(20))
print("Groups containing multiple families:", int(group_table["family_is_mixed"].sum()))
if manufacturer_col:
    print("Groups containing multiple manufacturers:", int((group_table["manufacturers"] > 1).sum()))
group_table.to_csv(OUTPUT_DIR / "insulation_group_composition_v1.csv")


,records,products,families,manufacturers,family_label,family_is_mixed
split_group_id,,,,,,
02/015/021/21,43,1,1,1,stone_wool,False
02/015/025/22,43,1,1,1,stone_wool,False
02/015/027/18,43,1,1,1,stone_wool,False
02/016/142/20,46,1,1,1,glass_wool,False
02/016/144/21,53,1,1,1,glass_wool,False
02/016/148/23,41,1,1,1,glass_wool,False
02/016/156/24,35,1,1,1,glass_wool,False
02/018/098/27,22,1,1,1,glass_wool,False
02/018/100/35,17,1,1,1,glass_wool,False


Groups containing multiple families: 0
Groups containing multiple manufacturers: 1


**Conclusion:** if groups contain multiple families, stratification must use a documented group-level label or a multi-label approach. Do not split the same group across train, validation and test to improve class balance.

## 7. Simulate group-aware split candidates

The following simulation uses `GroupShuffleSplit` on group IDs. It is a diagnostic, not an automatic final split. It reports record, product and group counts and family coverage for each candidate. Because rare families may have only one group, no split can guarantee representation of every family without leakage.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

# One row per independent group, with its dominant family label.
group_frame = (
    df.groupby("split_group_id")
      .agg(
          family=("material_family", lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN"),
          records=("variant_id", "count"),
          products=("product_id", "nunique")
      )
      .reset_index()
)

def simulate_group_split(test_size, validation_size, seed):
    # First hold out test groups.
    first = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    dev_idx, test_idx = next(first.split(group_frame, groups=group_frame["split_group_id"]))
    dev_groups = group_frame.iloc[dev_idx].copy()
    test_groups = group_frame.iloc[test_idx].copy()

    # Then hold out validation groups from the remaining development pool.
    # validation_size is expressed as a fraction of the original group pool.
    relative_val = validation_size / (1 - test_size)
    if not 0 < relative_val < 1:
        raise ValueError("Invalid validation fraction relative to development pool.")
    second = GroupShuffleSplit(n_splits=1, test_size=relative_val, random_state=seed + 1000)
    train_idx, val_idx = next(
        second.split(dev_groups, groups=dev_groups["split_group_id"])
    )
    train_groups = dev_groups.iloc[train_idx].copy()
    val_groups = dev_groups.iloc[val_idx].copy()

    assignment = {}
    for split_name, part in [
        ("train", train_groups), ("validation", val_groups), ("test", test_groups)
    ]:
        for gid in part["split_group_id"]:
            assignment[gid] = split_name

    row_assign = df[["split_group_id", "product_id", "variant_id", "material_family"]].copy()
    row_assign["candidate_split"] = row_assign["split_group_id"].map(assignment)

    summary = (
        row_assign.groupby("candidate_split")
        .agg(records=("variant_id", "count"),
             products=("product_id", "nunique"),
             groups=("split_group_id", "nunique"),
             families=("material_family", "nunique"))
    )
    family_coverage = (
        row_assign.groupby(["material_family", "candidate_split"])["split_group_id"]
        .nunique().unstack(fill_value=0)
    )
    for name in ["train", "validation", "test"]:
        if name not in family_coverage.columns:
            family_coverage[name] = 0
    family_coverage = family_coverage[["train", "validation", "test"]]
    return row_assign, summary, family_coverage

candidate_specs = [
    {"name": "candidate_A_70_15_15", "test_size": 0.15, "validation_size": 0.15},
    {"name": "candidate_B_70_20_10", "test_size": 0.10, "validation_size": 0.20},
    {"name": "candidate_C_60_20_20", "test_size": 0.20, "validation_size": 0.20},
]

simulation_summaries = []
simulation_family_tables = {}
simulation_assignments = {}

for spec in candidate_specs:
    assignment, summary, family_cov = simulate_group_split(
        spec["test_size"], spec["validation_size"], seed=42
    )
    simulation_assignments[spec["name"]] = assignment
    simulation_family_tables[spec["name"]] = family_cov
    summary = summary.reset_index()
    summary.insert(0, "candidate", spec["name"])
    simulation_summaries.append(summary)

    print("\\n", spec["name"])
    display(summary)
    display(family_cov)

simulation_summary = pd.concat(simulation_summaries, ignore_index=True)
simulation_summary.to_csv(OUTPUT_DIR / "group_split_candidate_summaries_v1.csv", index=False)

for name, table in simulation_family_tables.items():
    table.to_csv(OUTPUT_DIR / f"{name}_family_group_coverage.csv")


\n candidate_A_70_15_15


,candidate,candidate_split,records,products,groups,families
0,candidate_A_70_15_15,test,299,11,11,9
1,candidate_A_70_15_15,train,1220,48,47,13
2,candidate_A_70_15_15,validation,155,11,11,4


candidate_split,train,validation,test
material_family,,,
EPS,6,0,0
PF,0,0,1
PIR,4,2,1
PUR,1,0,0
XPS,3,4,1
aerogel,1,0,0
cellulose,2,0,0
cork,0,0,1
glass_wool,18,4,2


\n candidate_B_70_20_10


,candidate,candidate_split,records,products,groups,families
0,candidate_B_70_20_10,test,167,7,7,6
1,candidate_B_70_20_10,train,1140,49,48,12
2,candidate_B_70_20_10,validation,367,14,14,9


candidate_split,train,validation,test
material_family,,,
EPS,6,0,0
PF,0,0,1
PIR,4,2,1
PUR,1,0,0
XPS,6,2,0
aerogel,1,0,0
cellulose,1,1,0
cork,1,0,0
glass_wool,18,4,2


\n candidate_C_60_20_20


,candidate,candidate_split,records,products,groups,families
0,candidate_C_60_20_20,test,404,14,14,9
1,candidate_C_60_20_20,train,927,41,41,12
2,candidate_C_60_20_20,validation,343,15,14,8


candidate_split,train,validation,test
material_family,,,
EPS,4,2,0
PF,0,0,1
PIR,3,2,2
PUR,1,0,0
XPS,6,1,1
aerogel,0,1,0
cellulose,1,1,0
cork,0,0,1
glass_wool,15,5,4


**Conclusion:** compare the candidate tables rather than selecting a split by its headline percentage. The most important checks are independent groups per split, product counts, family coverage and zero group overlap. Random group splitting can still omit rare families from validation or test; this is a genuine limitation of the current data.

## 8. Check leakage and family coverage for each candidate

In [8]:
for name, assignment in simulation_assignments.items():
    split_groups = {
        split_name: set(assignment.loc[
            assignment["candidate_split"].eq(split_name), "split_group_id"
        ])
        for split_name in ["train", "validation", "test"]
    }
    overlaps = {
        "train_validation": len(split_groups["train"] & split_groups["validation"]),
        "train_test": len(split_groups["train"] & split_groups["test"]),
        "validation_test": len(split_groups["validation"] & split_groups["test"]),
    }
    print(name, "group overlaps:", overlaps)
    assert all(v == 0 for v in overlaps.values()), "Group leakage detected"

    family_coverage = simulation_family_tables[name]
    absent_test = family_coverage.index[family_coverage["test"].eq(0)].tolist()
    absent_validation = family_coverage.index[family_coverage["validation"].eq(0)].tolist()
    print("Families absent from test:", absent_test)
    print("Families absent from validation:", absent_validation)


candidate_A_70_15_15 group overlaps: {'train_validation': 0, 'train_test': 0, 'validation_test': 0}
Families absent from test: ['EPS', 'PUR', 'aerogel', 'cellulose', 'glass_wool_loosefill', 'recycled_cotton', 'wood_wool']
Families absent from validation: ['EPS', 'PF', 'PUR', 'aerogel', 'cellulose', 'cork', 'glass_wool_loosefill', 'polyester_wool', 'recycled_cotton', 'stone_wool', 'wood_wool', 'wood_wool_composite']
candidate_B_70_20_10 group overlaps: {'train_validation': 0, 'train_test': 0, 'validation_test': 0}
Families absent from test: ['EPS', 'PUR', 'XPS', 'aerogel', 'cellulose', 'cork', 'glass_wool_loosefill', 'recycled_cotton', 'wood_wool', 'wood_wool_composite']
Families absent from validation: ['EPS', 'PF', 'PUR', 'aerogel', 'cork', 'polyester_wool', 'wood_wool']
candidate_C_60_20_20 group overlaps: {'train_validation': 0, 'train_test': 0, 'validation_test': 0}
Families absent from test: ['EPS', 'PUR', 'aerogel', 'cellulose', 'glass_wool_loosefill', 'recycled_cotton', 'wood_wo

**Conclusion:** zero overlap is mandatory, but it is not sufficient. A split can be leakage-safe and still be unsuitable for broad family-level evaluation because some families are absent or represented by too few groups.

## 9. Recommended evaluation design

Use this as the proposed protocol for the next modelling phase, subject to review of the simulation outputs:

1. Preserve the group as the indivisible split unit.
2. Use grouped cross-validation within the development data for model comparison and tuning.
3. Keep a final product-group holdout untouched until the model and hyperparameters are selected.
4. Do not claim performance for families that have no independent test groups.
5. For rare families, collect more independent products or mark them unsupported/exploratory.
6. Report records, products, groups, manufacturers and family composition for every split.
7. For conductivity regression, check target distribution by group and product; avoid leakage from conductivity, thermal resistance and thickness as inputs.
8. Keep evidence eligibility explicit. Do not silently mix verified, partially verified and derived values.

## 10. Export a structured data collection template

This template is intentionally blank. Populate it only with real products and traceable source evidence.

In [9]:
collection_template = pd.DataFrame(columns=[
    "product_id_new",
    "material_family",
    "product_name",
    "manufacturer",
    "independent_product_group_id",
    "variant_or_thickness_range",
    "thermal_conductivity_W_mK",
    "thickness_mm",
    "thermal_resistance_m2K_W",
    "density_kg_m3",
    "specific_heat_J_kgK",
    "compressive_strength_kPa",
    "vapour_resistance_factor_mu",
    "tensile_strength_perpendicular_kPa",
    "mass_per_area_kg_m2",
    "gwp_a1_a3_kgco2e_m2",
    "reaction_to_fire",
    "data_status",
    "evidence_type",
    "source_document",
    "source_url_or_reference",
    "document_date",
    "units_and_test_method",
    "reviewer_notes"
])
template_path = OUTPUT_DIR / "new_insulation_product_collection_template_v1.csv"
collection_template.to_csv(template_path, index=False)
print("Blank collection template exported:", template_path)


Blank collection template exported: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI\06_AI_Performance_Engine\02_ML_Engine\04_Evaluation\reports\insulation_expansion_split_design_v1\new_insulation_product_collection_template_v1.csv


**Conclusion:** one row should represent the agreed product/variant observation level in the data model. Before appending collected data to the master dataset, reconcile product IDs, variant IDs, group IDs, units and provenance rules with the existing schema.

## 11. Final checks and exports

In [10]:
print("Output directory:", OUTPUT_DIR)
print("\\nExported files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)

print("\\nSource dataset rows:", len(df_all))
print("Insulation rows:", len(df))
print("Products:", df["product_id"].nunique())
print("Groups:", df["split_group_id"].nunique())
print("Families:", df["material_family"].nunique())
print("Model training performed: NO")
print("Master dataset modified: NO")


Output directory: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI\06_AI_Performance_Engine\02_ML_Engine\04_Evaluation\reports\insulation_expansion_split_design_v1
\nExported files:
- candidate_A_70_15_15_family_group_coverage.csv
- candidate_B_70_20_10_family_group_coverage.csv
- candidate_C_60_20_20_family_group_coverage.csv
- existing_insulation_product_research_checklist_v1.csv
- group_split_candidate_summaries_v1.csv
- insulation_family_collection_targets_v1.csv
- insulation_group_composition_v1.csv
- insulation_property_collection_priorities_v1.csv
- new_insulation_product_collection_template_v1.csv
\nSource dataset rows: 1941
Insulation rows: 1674
Products: 70
Groups: 69
Families: 16
Model training performed: NO
Master dataset modified: NO


## Final conclusion

The current insulation dataset contains many variant records but substantially fewer independent products and product groups. The expansion plan should therefore prioritise **new, traceable independent products**, especially in families with very few groups, rather than simply adding more variants to products already represented.

The split simulations are diagnostic only. Choose a final evaluation strategy after reviewing the actual candidate outputs and agreeing the intended classification scope. No split can manufacture missing family support, and no random row split should be used to inflate apparent performance.

**Next decision after running this notebook:** review the candidate split tables and agree which insulation families are in scope for the first supervised model experiment.